
# 03 — Context-conditional allocation

The contribution that is most identifiably new, and it follows from one property of
the data that nobody had looked at: **the context flags act on the variance of demand,
not its level** (notebook 00).

A marginally calibrated allocator therefore hits its target *on average* while
systematically under-provisioning when the network is under stress — the failures
cluster exactly where they hurt most. The claim tested here:

> Marginal conformal calibration misses its nominal rate inside the elevated-risk
> context group; locally adaptive and Mondrian calibration close that gap at
> near-identical capacity cost.

This is falsifiable, and it is reported in both directions: it **holds on GP** and is
**correctly null on Robi**, which has no flag with elevated variance.

In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# Works in Colab and locally. In Colab, clone the repo first:
#     !git clone <repo-url> bwalloc && %cd bwalloc
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "src" / "bwalloc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

try:
    import xgboost  # noqa: F401
except ImportError:
    !pip install -q xgboost

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import bwalloc as bw
from bwalloc.plots import use_paper_style

bw.set_seed()
use_paper_style()
pd.set_option("display.width", 200)
RESULTS = ROOT / "experiments" / "results"
FIGURES = ROOT / "paper" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("bwalloc", bw.__version__, "| results:", RESULTS)

In [ ]:

from bwalloc.context import assign_binary_groups, flag_report, group_sizes
from bwalloc.data import load, sampling_profile
from bwalloc.features import FeatureConfig, build_features
from bwalloc.models import xgboost_point
from bwalloc.pipeline import coverage_table, run_allocation_backtest
from bwalloc.splits import rolling_origin

OPERATOR = "gp"
df = load(OPERATOR)
profile = sampling_profile(df)

# Which flags mark elevated *uncertainty*? Levene's test on detrended residuals.
report = flag_report(df, OPERATOR)
report[["flag", "n_flagged", "mean_diff", "p_level", "resid_sd_flagged",
        "resid_sd_other", "variance_ratio", "p_variance"]]


### Group construction, and why it is a two-group split

The raw flags overlap (`rain ∩ gathering` = 33 rows), and Mondrian calibration needs
*disjoint* groups. A four-group split leaves the gathering group with ~20 calibration
residuals, and a group of *m* residuals cannot express a level finer than `1/(m+1)` —
τ = 0.99 needs 99. So the split is two groups: `elevated_risk = rain ∪ gathering`
against `baseline`. Both merged groups are the elevated-variance ones, so the merge is
principled rather than convenient.

`conformal.min_calibration_size` enforces this and **raises rather than clipping**.
Silently returning the largest residual would report a guarantee the data cannot
support — a subtler version of the error this project exists to correct.

In [ ]:

from bwalloc.conformal import min_calibration_size

groups = assign_binary_groups(df).reindex(build_features(df, profile, FeatureConfig())[0].index)
print(group_sizes(groups).to_string(index=False), "\n")
for tau in (0.80, 0.90, 0.95, 0.99):
    print(f"tau={tau:.2f} needs at least {min_calibration_size(tau):3d} calibration residuals")

## The result

Coverage by context group, pooled across folds, weighted by test-block size.

In [ ]:

X, y = build_features(df, profile, FeatureConfig())
folds = rolling_origin(len(y), n_folds=8, calib_frac=0.30)
per_fold, allocations = run_allocation_backtest(
    X, y, folds, groups, model_factory=xgboost_point, taus=(0.80, 0.90, 0.95),
)
coverage_table(per_fold, "coverage")

In [ ]:

# The capacity cost of closing the gap.
coverage_table(per_fold, "mean_allocation_ratio")

In [ ]:

from bwalloc.plots import plot_coverage_by_group

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, tau in zip(axes, (0.80, 0.90, 0.95)):
    plot_coverage_by_group(per_fold, tau=tau, ax=ax)
    ax.set_title(f"tau = {tau:.2f}")
fig.tight_layout()
fig.savefig(FIGURES / f"fig4_coverage_by_group_{OPERATOR}.png", dpi=200, bbox_inches="tight")


## Reading the result, and the null on Robi

On GP the gap is large and consistent: at τ = 0.95, marginal calibration delivers
0.963 on the baseline group but **0.866** inside the elevated-risk group. Locally
adaptive calibration lifts that to 0.911 and Mondrian to 0.922.

Now switch `OPERATOR` to `"robi"` and re-run. On Robi **no flag has elevated
variance** — `is_powercut` and `is_event` actually mark *calmer* periods — and the
method correspondingly does nothing (τ = 0.95 elevated: marginal 0.889, adaptive
0.875, Mondrian 0.903).

That is the right outcome to report. The claim is not "context-conditional
calibration always helps"; it is "context-conditional calibration helps exactly when
context carries variance signal, and can be tested for in advance". The null on Robi
is what makes it a prediction rather than a story.

**Caveat to keep in the paper.** The elevated-risk group carries ~40 calibration
residuals, so per-group coverage estimates are reported with Clopper–Pearson
intervals and those intervals are wide. Robi carries five of the nine flags and no
rain annotation at all, and its group is 89 rows — directional only.